In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.models import run_isolation_forest, run_lof, run_pca, run_random_forest

X = np.load('../data/features/X_full.npy')
X_counts = np.load('../data/features/X_counts_full.npy')
y = np.load('../data/features/y_full.npy')

print("Shape:", X.shape, "| Anomalies:", y.sum(), f"({y.mean()*100:.2f}%)")

Shape: (575061, 45) | Anomalies: 16838 (2.93%)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

X_train_normal = X_train[y_train == 0]
contamination = float(y.mean())
rng = np.random.default_rng(42)
lof_idx = rng.choice(len(X_train_normal), size=50_000, replace=False)
X_train_lof = X_train_normal[lof_idx]

print("Train normal:", X_train_normal.shape, "| LOF subsample:", X_train_lof.shape)
print("Test:", X_test.shape, "| Anomalies in test:", y_test.sum())

Train normal: (390755, 45) | LOF subsample: (50000, 45)
Test: (172519, 45) | Anomalies in test: 5051


In [3]:
results = []
results.append(run_isolation_forest(X_train_normal, X_test, y_test, contamination))
results.append(run_pca(X_train_normal, X_test, y_test, contamination))
results.append(run_lof(X_train_lof, X_test, y_test, contamination))
results.append(run_random_forest(X_train, y_train, X_test, y_test))

tfidf_results = pd.DataFrame(results)
tfidf_results['Features'] = 'TF-IDF'
tfidf_results.sort_values('F1', ascending=False)

,Model,Precision,Recall,F1,ROC_AUC,PR_AUC,FPR,TP,FP,FN,TN,TrainTime,InferTime,Features
3,Random Forest,0.9959,0.9992,0.9975,0.9999,0.9997,0.0001,5047,21,4,167447,26.883,1.009,TF-IDF
1,PCA,0.9184,0.9113,0.9148,0.9993,0.9847,0.0024,4603,409,448,167059,0.080,0.104,TF-IDF
2,Local Outlier Factor,0.8347,1.0000,0.9099,0.9992,0.9739,0.0060,5051,1000,0,166468,4.783,9.855,TF-IDF
0,Isolation Forest,0.4082,0.6252,0.4939,0.9519,0.2931,0.0273,3158,4579,1893,162889,2.593,0.892,TF-IDF


In [4]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_counts, y, test_size=0.3, stratify=y, random_state=42)

Xc_train_normal = Xc_train[yc_train == 0]
Xc_train_lof = Xc_train_normal[lof_idx]

results_counts = []
results_counts.append(run_isolation_forest(Xc_train_normal, Xc_test, yc_test, contamination))
results_counts.append(run_pca(Xc_train_normal, Xc_test, yc_test, contamination))
results_counts.append(run_lof(Xc_train_lof, Xc_test, yc_test, contamination))
results_counts.append(run_random_forest(Xc_train, yc_train, Xc_test, yc_test))

count_results = pd.DataFrame(results_counts)
count_results['Features'] = 'Raw counts'
count_results.sort_values('F1', ascending=False)

,Model,Precision,Recall,F1,ROC_AUC,PR_AUC,FPR,TP,FP,FN,TN,TrainTime,InferTime,Features
3,Random Forest,0.9966,0.9994,0.9980,1.0000,0.9999,0.0001,5048,17,3,167451,43.608,1.207,Raw counts
1,PCA,0.9454,0.9355,0.9404,0.9997,0.9866,0.0016,4725,273,326,167195,0.094,0.118,Raw counts
2,Local Outlier Factor,0.8405,0.9998,0.9133,0.9996,0.9795,0.0057,5050,958,1,166510,3.288,9.930,Raw counts
0,Isolation Forest,0.0780,0.0788,0.0784,0.6920,0.0510,0.0281,398,4707,4653,162761,3.024,1.131,Raw counts


In [5]:
comparison = pd.concat([tfidf_results, count_results], ignore_index=True)
comparison.to_csv('../results/comparison_full.csv', index=False)
comparison.sort_values('F1', ascending=False)

,Model,Precision,Recall,F1,ROC_AUC,PR_AUC,FPR,TP,FP,FN,TN,TrainTime,InferTime,Features
7,Random Forest,0.9966,0.9994,0.9980,1.0000,0.9999,0.0001,5048,17,3,167451,43.608,1.207,Raw counts
3,Random Forest,0.9959,0.9992,0.9975,0.9999,0.9997,0.0001,5047,21,4,167447,26.883,1.009,TF-IDF
5,PCA,0.9454,0.9355,0.9404,0.9997,0.9866,0.0016,4725,273,326,167195,0.094,0.118,Raw counts
1,PCA,0.9184,0.9113,0.9148,0.9993,0.9847,0.0024,4603,409,448,167059,0.080,0.104,TF-IDF
6,Local Outlier Factor,0.8405,0.9998,0.9133,0.9996,0.9795,0.0057,5050,958,1,166510,3.288,9.930,Raw counts
2,Local Outlier Factor,0.8347,1.0000,0.9099,0.9992,0.9739,0.0060,5051,1000,0,166468,4.783,9.855,TF-IDF
0,Isolation Forest,0.4082,0.6252,0.4939,0.9519,0.2931,0.0273,3158,4579,1893,162889,2.593,0.892,TF-IDF
4,Isolation Forest,0.0780,0.0788,0.0784,0.6920,0.0510,0.0281,398,4707,4653,162761,3.024,1.131,Raw counts
